# Lookback Macro Metrics (Mean ± Std) by Variant

1. Set `FILE_SPECS` with one or more `lookback_search_results.csv` per variant.
2. Use `variant="tagi_v"` for TAGI-V runs and `variant="without_tagi_v"` for baseline runs.
3. Run all cells to compare both variants by lookback using mean ± std error bars.
4. Each macro metric is plotted in its own standalone figure.
5. The notebook returns seed coverage, best value per (variant, metric, seed), and winner seed per (variant, metric).



In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd

# Plotting defaults used across experiment scripts
mpl.rcParams.update(
    {
        "pgf.texsystem": "pdflatex",
        "font.family": "serif",
        "text.usetex": False,
        "pgf.rcfonts": False,
        "pgf.preamble": r"\usepackage{amsfonts}\usepackage{amssymb}\usepackage{amsmath}",
        "lines.linewidth": 1,
    }
)



In [ ]:
# Add one or more files per variant (baseline, TAGI-V, embeddings, etc.)
FILE_SPECS = [
    # {
    #     "path": "/home/dw/cuTAGI_DW/out/seed2016/train_use_100/experiment01_global_no-embeddings_gridsearch/lookback_search_results.csv",
    #     "variant": "without_tagi_v",
    # },
    {
        "path": "/home/dw/cuTAGI_DW/out/seed2016/train_use_100/experiment01_global_no-embeddings_gridsearch_tagiv/lookback_search_results.csv",
        "variant": "tagi_v",
    },
    {
        "path": "/home/dw/cuTAGI_DW/out/seed2016/train_use_100/experiment01_global_simple-embeddings_gridsearch_tagiv/lookback_search_results.csv",
        "variant": "tagi_v_simple_embeddings",
    },
    # {
    #     "path": "/home/dw/cuTAGI_DW/out/seed2012/train_use_100/experiment01_global_no-embeddings_gridsearch/lookback_search_results.csv",
    #     "variant": "without_tagi_v",
    # },
    {
        "path": "/home/dw/cuTAGI_DW/out/seed2012/train_use_100/experiment01_global_no-embeddings_gridsearch_tagiv/lookback_search_results.csv",
        "variant": "tagi_v",
    },
    # {
    #     "path": "/home/dw/cuTAGI_DW/out/seed2005/train_use_100/experiment01_global_no-embeddings_gridsearch/lookback_search_results.csv",
    #     "variant": "without_tagi_v",
    # },
    {
        "path": "/home/dw/cuTAGI_DW/out/seed2005/train_use_100/experiment01_global_no-embeddings_gridsearch_tagiv/lookback_search_results.csv",
        "variant": "tagi_v",
    },
    {
        "path": "/home/dw/cuTAGI_DW/out/seed2005/train_use_100/experiment01_global_simple-embeddings_gridsearch_tagiv/lookback_search_results.csv",
        "variant": "tagi_v_simple_embeddings",
    },
    {
        "path": "/home/dw/cuTAGI_DW/out/seed2012/train_use_100/experiment01_global_simple-embeddings_gridsearch_tagiv/lookback_search_results.csv",
        "variant": "tagi_v_simple_embeddings",
    },
        {
        "path": "/home/dw/cuTAGI_DW/out/seed2005/train_use_100/experiment01_global_hierarchical-embeddings_gridsearch_tagiv/lookback_search_results.csv",
        "variant": "tagi_v_hier_embeddings",
    },
    {
        "path": "/home/dw/cuTAGI_DW/out/seed2012/train_use_100/experiment01_global_hierarchical-embeddings_gridsearch_tagiv/lookback_search_results.csv",
        "variant": "tagi_v_hier_embeddings",
    },
        {
        "path": "/home/dw/cuTAGI_DW/out/seed2016/train_use_100/experiment01_global_hierarchical-embeddings_gridsearch_tagiv/lookback_search_results.csv",
        "variant": "tagi_v_hier_embeddings",
    },
]

LOOKBACK_COLUMN = "lookback"
MACRO_PREFIX = "macro_"



In [ ]:
import re

frames = []
macro_sets = []


def infer_seed(path: Path, df: pd.DataFrame) -> str:
    path_match = re.search(r"(seed\d+)", str(path), flags=re.IGNORECASE)
    if path_match:
        return path_match.group(1).lower()

    if "experiment_name" in df.columns and df["experiment_name"].notna().any():
        joined_names = " ".join(df["experiment_name"].astype(str).tolist())
        exp_match = re.search(r"(seed\d+)", joined_names, flags=re.IGNORECASE)
        if exp_match:
            return exp_match.group(1).lower()

    return "unknown_seed"


for i, spec in enumerate(FILE_SPECS, start=1):
    if "path" not in spec or "variant" not in spec:
        raise ValueError(
            "Each FILE_SPECS entry must include both 'path' and 'variant'."
        )

    p = Path(spec["path"]).expanduser()
    variant = str(spec["variant"]).strip()
    if not variant:
        raise ValueError(f"Variant cannot be empty for entry #{i}: {spec}")

    if not p.exists():
        raise FileNotFoundError(f"File not found: {p}")

    df = pd.read_csv(p)
    if LOOKBACK_COLUMN not in df.columns:
        raise ValueError(f"{p} is missing required column: {LOOKBACK_COLUMN}")

    macro_cols = [c for c in df.columns if c.startswith(MACRO_PREFIX)]
    if not macro_cols:
        raise ValueError(f"{p} has no columns starting with '{MACRO_PREFIX}'")

    macro_sets.append(set(macro_cols))

    grouped = (
        df[[LOOKBACK_COLUMN] + macro_cols]
        .groupby(LOOKBACK_COLUMN, as_index=False)
        .mean(numeric_only=True)
        .sort_values(LOOKBACK_COLUMN)
    )
    grouped["variant"] = variant
    grouped["source"] = f"path_{i}"
    grouped["source_name"] = p.parent.name or p.stem or f"path_{i}"
    grouped["seed"] = infer_seed(p, df)

    frames.append(grouped)

if not frames:
    raise ValueError("FILE_SPECS is empty.")

macro_metrics = sorted(set.intersection(*macro_sets))
if not macro_metrics:
    raise ValueError("No shared macro metrics were found across all files")

combined = pd.concat(frames, ignore_index=True)
variants = sorted(combined["variant"].dropna().unique().tolist())
if len(variants) < 2:
    print("[warn] Only one variant found. Add both 'tagi_v' and 'without_tagi_v' for a comparison plot.")

summary_by_metric = {}
for metric in macro_metrics:
    stats = (
        combined.groupby(["variant", LOOKBACK_COLUMN])[metric]
        .agg(mean="mean", std=lambda s: s.std(ddof=0), count="count")
        .reset_index()
        .sort_values(["variant", LOOKBACK_COLUMN])
    )
    summary_by_metric[metric] = stats

summary_table = pd.concat(
    [
        summary_by_metric[m]
        .assign(metric=m)
        .rename(columns={"mean": "average"})[
            ["metric", "variant", LOOKBACK_COLUMN, "average", "std", "count"]
        ]
        for m in macro_metrics
    ],
    ignore_index=True,
)

per_seed_metric_rows = []
for metric in macro_metrics:
    metric_lower = metric.lower().replace("-", "_")
    higher_is_better = "log_lik" in metric_lower or "loglik" in metric_lower

    for (variant, seed), group in combined.groupby(["variant", "seed"]):
        metric_group = group[[LOOKBACK_COLUMN, metric, "source_name"]].dropna(subset=[metric])
        if metric_group.empty:
            continue

        metric_group = metric_group.sort_values(
            [metric, LOOKBACK_COLUMN],
            ascending=[not higher_is_better, True],
        )
        best_row = metric_group.iloc[0]

        per_seed_metric_rows.append(
            {
                "metric": metric,
                "variant": variant,
                "seed": seed,
                "best_lookback": int(best_row[LOOKBACK_COLUMN]),
                "metric_value": float(best_row[metric]),
                "source_name": best_row["source_name"],
                "direction": "higher_is_better" if higher_is_better else "lower_is_better",
            }
        )

per_seed_metric_table = pd.DataFrame(per_seed_metric_rows)
if per_seed_metric_table.empty:
    raise ValueError("Could not compute seed-level metric table from the provided files.")

best_seed_rows = []
for metric in macro_metrics:
    metric_lower = metric.lower().replace("-", "_")
    higher_is_better = "log_lik" in metric_lower or "loglik" in metric_lower

    metric_rows = per_seed_metric_table[per_seed_metric_table["metric"] == metric].copy()
    metric_rows = metric_rows.sort_values(
        ["variant", "metric_value", "best_lookback", "seed"],
        ascending=[True, not higher_is_better, True, True],
    )
    winners = metric_rows.groupby("variant", as_index=False).first()
    best_seed_rows.append(winners)

best_seed_by_variant_metric = pd.concat(best_seed_rows, ignore_index=True).sort_values(
    ["metric", "variant"]
)

print("Seed coverage by variant")
display(
    combined[["variant", "seed", "source_name"]]
    .drop_duplicates()
    .sort_values(["variant", "seed", "source_name"])
    .reset_index(drop=True)
)

print("Best value per (variant, metric, seed)")
display(
    per_seed_metric_table.sort_values(["metric", "variant", "seed"]).reset_index(drop=True)
)

print("Winner seed per (variant, metric)")
display(best_seed_by_variant_metric.reset_index(drop=True))

print("Aggregated variant-level summary by lookback")
display(summary_table.sort_values(["metric", "variant", LOOKBACK_COLUMN]).reset_index(drop=True))




In [ ]:
if len(variants) <= 10:
    palette = list(plt.get_cmap("tab10").colors)
else:
    palette = list(plt.get_cmap("tab20").colors)
color_map = {variant: palette[i % len(palette)] for i, variant in enumerate(variants)}

for metric in macro_metrics:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    stats = summary_by_metric[metric]

    for variant in variants:
        vstats = stats[stats["variant"] == variant].sort_values(LOOKBACK_COLUMN)
        if vstats.empty:
            continue

        ax.errorbar(
            vstats[LOOKBACK_COLUMN],
            vstats["mean"],
            yerr=vstats["std"],
            fmt="o-",
            color=color_map[variant],
            linewidth=2,
            markersize=5,
            capsize=4,
            capthick=1,
            elinewidth=1,
            label=f"{variant} (n={int(vstats['count'].max())})",
        )

    metric_lower = metric.lower().replace("-", "_")
    is_loglik = "log_lik" in metric_lower or "loglik" in metric_lower
    direction_text = "↑ higher is better" if is_loglik else "↓ lower is better"

    ax.set_title(metric)
    ax.set_xlabel(LOOKBACK_COLUMN)
    ax.set_ylabel("mean ± std")
    ax.grid(True, alpha=0.3)
    ax.text(
        0.02,
        0.96,
        direction_text,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=10,
        bbox={"facecolor": "white", "alpha": 0.7, "edgecolor": "none"},
    )
    ax.legend(loc="best", frameon=True)

    fig.tight_layout()
    plt.show()

